# 00 - Cryptic IP Pipeline: Colab Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tommaso-R-Marena/cryptic-ip-pipeline/blob/main/notebooks/00_colab_quickstart.ipynb)

This notebook bootstraps a fresh Google Colab runtime, installs the
`crypticip` package, verifies the CLI, runs the test suite, and walks
through a synthetic mini-proteome screen so you can confirm the
plumbing end-to-end **without** any large downloads or external
binaries.

Expected runtime: ~5 minutes. Disk: <100 MB.


## Run this first — fresh-Colab setup

Configure the variables below, then run every cell in this section in
order. Re-running is safe: the clone is updated in place and pip
installs are idempotent.

- `REPO_URL` / `BRANCH` — where to fetch the code from.
- `PROJECT_DIR` — where to clone into (`/content/...` lives on the
  Colab VM and is wiped between sessions).
- `MOUNT_DRIVE` — set `True` to mount Google Drive so outputs persist.
- `DRIVE_RESULTS` — if mounting, where on Drive to put `results/`.


In [ ]:
REPO_URL    = 'https://github.com/Tommaso-R-Marena/cryptic-ip-pipeline.git'
BRANCH      = 'main'
PROJECT_DIR = '/content/cryptic-ip-pipeline'
MOUNT_DRIVE = False                  # True to mount Google Drive
DRIVE_ROOT  = '/content/drive'
DRIVE_RESULTS = '/content/drive/MyDrive/crypticip/results'  # used only if MOUNT_DRIVE


In [ ]:
import os, subprocess, sys, pathlib

project = pathlib.Path(PROJECT_DIR)
project.parent.mkdir(parents=True, exist_ok=True)

def sh(cmd, check=True):
    print('$', ' '.join(cmd))
    return subprocess.run(cmd, check=check)

if (project / '.git').exists():
    sh(['git', '-C', str(project), 'fetch', '--quiet', 'origin', BRANCH])
    sh(['git', '-C', str(project), 'checkout', BRANCH])
    sh(['git', '-C', str(project), 'reset', '--hard', f'origin/{BRANCH}'])
else:
    sh(['git', 'clone', '--quiet', '--branch', BRANCH, REPO_URL, str(project)])

os.chdir(project)
print('cwd:', os.getcwd())
sh([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])
sh([sys.executable, '-m', 'pip', 'install', '-q', 'nbformat'])


In [ ]:
# Optional: mount Drive and redirect results/ onto it so outputs survive
# a runtime swap. Safe to re-run; pre-existing local results/ is backed
# up to a timestamped sibling rather than deleted.
import datetime, pathlib, shutil

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount(DRIVE_ROOT)
    drive_path = pathlib.Path(DRIVE_RESULTS)
    drive_path.mkdir(parents=True, exist_ok=True)
    target = pathlib.Path(PROJECT_DIR) / 'results'
    if target.is_symlink():
        target.unlink()
    elif target.exists():
        backup = target.with_name(f'results.local_backup_{datetime.datetime.now():%Y%m%d_%H%M%S}')
        shutil.move(str(target), str(backup))
        print('existing results/ backed up to:', backup)
    target.symlink_to(drive_path, target_is_directory=True)
    print('results/ ->', drive_path)
else:
    print('MOUNT_DRIVE=False; outputs will live on the Colab VM only.')


In [ ]:
# Verify the CLI is wired up correctly.
!crypticip --version
!crypticip check-env
!crypticip list-configs


## Run the test suite

All tests should pass; notebook-validation tests live in `tests/test_notebooks.py`.


In [ ]:
!python -m pytest -q


## Synthetic mini-proteome smoke workflow

Builds a 3-file 'fake yeast' proteome (using the test fixture
`tests/fixtures/tiny.pdb`) under `/content/crypticip_smoke/`, then runs
`index-proteome`, `screen`, `report`, `pymol`, and `experimental-plan`
against it. No network and no external binaries required.


In [ ]:
import shutil, pathlib, yaml

SMOKE = pathlib.Path('/content/crypticip_smoke')
PROT  = SMOKE / 'data/proteomes/yeast'
PROT.mkdir(parents=True, exist_ok=True)

# Three copies of the tiny test fixture, named like AlphaFold PDBs.
src = pathlib.Path('tests/fixtures/tiny.pdb')
assert src.exists(), 'tests/fixtures/tiny.pdb missing — wrong cwd?'
for i, uniprot in enumerate(['P00001', 'P00002', 'P00003']):
    shutil.copyfile(src, PROT / f'AF-{uniprot}-F1-model_v4.pdb')

cfg_path = SMOKE / 'smoke_config.yaml'
cfg_path.write_text(yaml.safe_dump({
    'paths': {
        'data_dir':      str(SMOKE / 'data'),
        'proteomes_dir': str(SMOKE / 'data' / 'proteomes'),
        'cache_dir':     str(SMOKE / 'cache'),
        'results_dir':   str(SMOKE / 'results'),
        'reports_dir':   str(SMOKE / 'results' / 'reports'),
        'screening_dir': str(SMOKE / 'results' / 'screening'),
        'experimental_dir': str(SMOKE / 'results' / 'experimental'),
    },
}))
print('smoke config:', cfg_path)


In [ ]:
!crypticip index-proteome --organism yeast --config /content/crypticip_smoke/smoke_config.yaml


In [ ]:
!crypticip screen --organism yeast --mode fast --workers 1 --limit 3 --config /content/crypticip_smoke/smoke_config.yaml


In [ ]:
!crypticip report --organism yeast --config /content/crypticip_smoke/smoke_config.yaml


In [ ]:
!crypticip pymol --organism yeast --top 5 --config /content/crypticip_smoke/smoke_config.yaml


In [ ]:
!crypticip experimental-plan --organism yeast --top 5 --config /content/crypticip_smoke/smoke_config.yaml


## What to download from Colab

The smoke run writes everything under `/content/crypticip_smoke/results/`.
The cell below zips the directory and triggers a browser download. If
you mounted Drive, outputs are already persisted there.


In [ ]:
import shutil, pathlib
out = shutil.make_archive('/content/crypticip_smoke_results', 'zip',
                          '/content/crypticip_smoke/results')
print('zipped to:', out, 'MB:', round(pathlib.Path(out).stat().st_size / 1e6, 2))
try:
    from google.colab import files
    files.download(out)
except Exception as e:
    print('skipping files.download (not on Colab):', e)


## What you just saw

- The repo cloned, the package installed, and the CLI is on PATH.
- Tests pass (excluding any that require external binaries).
- The CLI iterates over the synthetic proteome. Because **fpocket is
  not installed on stock Colab**, `screen` records zero pockets — this
  is fallback mode, *not* a scientific result. To run a real screen
  with detected pockets, install fpocket/APBS/PyMOL via condacolab
  (see `01_validation_colab.ipynb`, section 'Install external
  scientific binaries').
